**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Blind Source Separation & ICA

The cocktail-party problem, actually solved: several microphones each hear a *mixture* of sources, and — knowing nothing about the mixing — we unmix them. The key is a beautiful statistical loophole: Gaussianity is the one thing mixing *increases*. Verified the only way that matters: recovered sources correlate ≈1 with the planted truth.

## 1. Pre-requisites

[Statistical SP](./Statistical_Signal_Processing.ipynb), [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S3, [Independence](../Intro_Math/Analysis/Independence.ipynb).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

# three planted sources: a chirp 'voice', a square-wave 'hum', an impulsive 'percussion'
fs, T_dur = 8000, 3.0
t = np.arange(0, T_dur, 1/fs)
s1 = sig.chirp(t, 300, T_dur, 800) * (1 + 0.3*np.sin(2*np.pi*2*t))
s2 = sig.square(2*np.pi*120*t) * 0.7
s3 = np.zeros_like(t)
for tc in rng.uniform(0, T_dur, 25):
    i = int(tc*fs); s3[i:i+150] += np.exp(-np.arange(150)/25) * rng.choice([-2, 2])
S_true = np.stack([s1, s2, s3])
S_true = (S_true - S_true.mean(1, keepdims=True)) / S_true.std(1, keepdims=True)

A_mix = rng.standard_normal((3, 3))                    # unknown room acoustics
X = A_mix @ S_true                                      # what the microphones record

---
### 🕐 Session 1 of 3 — *The Problem & Why Correlation Isn't Enough* (~35 min)
**Goal:** see mixing destroy the sources; understand why PCA/whitening only gets you halfway.
**Feeds into:** Session 2 (the non-Gaussian loophole).

---

## 2. Three Microphones, Three Tangles

💡 **Intuition.** Each mic hears $x_i = \sum_j a_{ij} s_j$: linear, instantaneous mixing. **Whitening** (decorrelating via the covariance [eigendecomposition](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb)) can undo mixing *up to a rotation* — but second-order statistics are **rotation-blind**: every rotation of white signals is equally white. Correlation has taken you to a sphere of candidate unmixings and gone silent. Something beyond variance must pick the rotation — that something is Session 2.

In [2]:
fig, axes = plt.subplots(2, 3, figsize=(10.5, 3.2))
for ax, s, name in zip(axes[0], S_true, ["'voice' (chirp)", "'hum' (square)", "'percussion'"]):
    ax.plot(t[:2000], s[:2000], linewidth=0.7); ax.set_title("source: " + name, fontsize=8)
for ax, x in zip(axes[1], X):
    ax.plot(t[:2000], x[:2000], linewidth=0.7, color="crimson"); ax.set_title("a microphone", fontsize=8)
plt.tight_layout(); plt.show()

# whiten
Xc = X - X.mean(1, keepdims=True)
C = Xc @ Xc.T / Xc.shape[1]
w_eig, V_eig = np.linalg.eigh(C)
W_white = np.diag(w_eig**-0.5) @ V_eig.T
Z = W_white @ Xc
print("after whitening, covariance = I:", np.allclose(Z @ Z.T / Z.shape[1], np.eye(3), atol=1e-10))
print("...but the sources are still mixed: |corr| of Z1 with each true source:",
      np.abs([np.corrcoef(Z[0], s)[0,1] for s in S_true]).round(2))

after whitening, covariance = I: True
...but the sources are still mixed: |corr| of Z1 with each true source: [0.72 0.63 0.28]


/tmp/ipykernel_2985363/4044920668.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 3 — *The Non-Gaussian Loophole & FastICA* (~40 min)
**Goal:** the CLT in reverse: mixtures are MORE Gaussian than sources — so maximize non-Gaussianity.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (limits & practice).

---

## 3. Gaussianity as a Compass

💡 **Intuition.** The [CLT](../Intro_Math/Analysis/Independence.ipynb) says sums of independent things drift *toward* Gaussian. Flip it around: each microphone (a sum of sources) is **more Gaussian than any single source** — so to unmix, rotate the whitened data until each output is as **non-Gaussian as possible**. That's all of ICA. FastICA does it with a fixed-point iteration on a smooth non-Gaussianity score (we use $\log\cosh$), one source at a time, deflating (Gram–Schmidt) so each new direction is orthogonal to the found ones. Built-in limits fall out of the logic: source order and sign/scale are unrecoverable, and **two Gaussian sources cannot be separated** (their mixtures are exactly rotation-symmetric).

In [3]:
def fastica(Z, n_comp, iters=300):
    W = np.zeros((n_comp, Z.shape[0]))
    for k in range(n_comp):
        w = rng.standard_normal(Z.shape[0]); w /= np.linalg.norm(w)
        for _ in range(iters):
            u = w @ Z
            g, gp = np.tanh(u), 1 - np.tanh(u)**2            # logcosh score & derivative
            w_new = (Z * g).mean(1) - gp.mean() * w
            for j in range(k):                                # deflate: stay ⊥ to found sources
                w_new -= (w_new @ W[j]) * W[j]
            w_new /= np.linalg.norm(w_new)
            if np.abs(np.abs(w_new @ w) - 1) < 1e-10: w = w_new; break
            w = w_new
        W[k] = w
    return W

W_ica = fastica(Z, 3)
S_hat = W_ica @ Z

# ORACLE: each recovered source must match ONE true source with |corr| ≈ 1
corr = np.abs(np.corrcoef(np.vstack([S_hat, S_true]))[:3, 3:])
print("correlation matrix (recovered × true):\n", corr.round(3))
best = corr.max(1)
print(f"per-source best |corr|: {best.round(4)}  — separation achieved: {bool((best > 0.98).all())}")
assert (best > 0.98).all() and len(set(corr.argmax(1))) == 3

correlation matrix (recovered × true):
 [[1.    0.015 0.   ]
 [0.003 0.035 1.   ]
 [0.008 0.999 0.008]]
per-source best |corr|: [1.     1.     0.9993]  — separation achieved: True


In [4]:
fig, axes = plt.subplots(3, 1, figsize=(9, 3.6), sharex=True)
order = corr.argmax(1)
for ax, sh, idx in zip(axes, S_hat, order):
    flip = np.sign(np.corrcoef(sh, S_true[idx])[0, 1])
    ax.plot(t[:2000], flip*sh[:2000], linewidth=0.7, label="recovered")
    ax.plot(t[:2000], S_true[idx][:2000], "k--", linewidth=0.5, alpha=0.6, label="truth")
    ax.legend(fontsize=6, loc="upper right")
plt.suptitle("unmixed blind — no knowledge of the mixing matrix was used")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2985363/187698282.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 3 — *Limits, Diagnostics & Practice* (~30 min)
**Goal:** what ICA can't do, how to sanity-check it, and where it runs in the wild.
**Builds on:** Session 2.

---

## 4. The Honest Fine Print

In [5]:
# the promised failure: two GAUSSIAN sources are unseparable — watch it happen
S_gauss = rng.standard_normal((2, 40000))
X_g = rng.standard_normal((2, 2)) @ S_gauss
Cg = X_g @ X_g.T / X_g.shape[1]
wg, Vg = np.linalg.eigh(Cg)
Zg = np.diag(wg**-0.5) @ Vg.T @ X_g
best_corrs = []
for trial in range(5):                                # multiple restarts — none will succeed
    Wg = fastica(Zg, 2, iters=200)
    cg = np.abs(np.corrcoef(np.vstack([Wg @ Zg, S_gauss]))[:2, 2:])
    best_corrs.append(cg.max(1).min())
print(f"Gaussian sources, worst-recovered |corr| across 5 restarts: {np.round(best_corrs, 3)}")
print("→ compare the non-Gaussian case: 0.999+ EVERY time. Here the answers scatter with the")
print("  random start — the rotation is unidentifiable, and restart-inconsistency is the tell.")

Gaussian sources, worst-recovered |corr| across 5 restarts: [0.784 0.779 0.76  0.968 0.972]
→ compare the non-Gaussian case: 0.999+ EVERY time. Here the answers scatter with the
  random start — the rotation is unidentifiable, and restart-inconsistency is the tell.


**Field guide.**

- **Works:** EEG artifact removal (eye blinks are gloriously non-Gaussian), [audio](./Audio_Speech_DSP.ipynb) unmixing with instantaneous mixtures, hyperspectral unmixing.
- **Fails or needs upgrades:** convolutive/reverberant mixing (rooms delay, not just scale — needs frequency-domain ICA), more sources than mics (underdetermined → [sparsity](./Sparse_Dictionary_Learning.ipynb) to the rescue), Gaussian-ish sources.
- **Diagnostics:** always check kurtosis of outputs (should be far from 0), and run restarts — consistent answers across restarts are the practical identifiability certificate.
- **Lineage:** [contrastive learning](../Intro_Mach_Learn/Representation_Learning.ipynb) and modern disentanglement research are ICA's descendants (nonlinear ICA is provably impossible without auxiliary structure — a live research frontier).

## 5. Conclusion

Whitening gets you to a rotation; the CLT-in-reverse picks it; FastICA computes it (recovered × truth correlations > 0.99, verified); and Gaussian sources mark the hard boundary of the possible (also verified). Blindness, it turns out, is negotiable — Gaussianity isn't.

---
## Where next

- [Array Processing](./Array_Processing.ipynb) — unmixing with *geometry* instead of statistics.
- [Representation Learning](../Intro_Mach_Learn/Representation_Learning.ipynb) — the neural descendants.
- [Manifold Optimization](../Intro_Math/Optimization/Manifold_Optimization.ipynb) — ICA's rotation search lives on the Stiefel manifold.